# Hexes upon hexes

In [ ]:
#| default_exp plot.chunk

In [ ]:
#| export
import sys
import math
import numpy as np
import math
from collections import namedtuple
from dataclasses import dataclass, field
from fastcore.basics import patch
from dataclasses import dataclass, field
from typing import Iterator
import numpy as np


In [ ]:
#| export
from HexMagic.plot.primitives import MapCord, MapSize, MapRect, MapPath, PrimitiveDemo
from HexMagic.plot.cube import HexPosition
from HexMagic.plot.hex import Hex, HexGrid

from HexMagic.styles import StyleCSS,  SVGBuilder


In [ ]:
#| export


class HexChunk:
    """A hex-shaped region of exactly N rings from center, with optional halo."""
    
    # Class-level cache: rings -> (spiral list, pos->idx dict)
    _spiral_cache: dict[int, tuple[list[HexPosition], dict[HexPosition, int]]] = {}

    def __init__(self,
        position: HexPosition,  # Chunk coordinate (not world)
        rings: int,
        style: StyleCSS = None,
        core_size: int = None):  # If None, all hexes are core

        self.position = position
        self.rings = rings
        self.style = style or StyleCSS("chunk_default", fill="#f0f0f0", stroke="#ccc", stroke_width=1)
        
        # Build/retrieve spiral mapping
        if rings not in HexChunk._spiral_cache:
            spiral = HexPosition.origin().spiral(rings)
            pos_to_idx = {pos: i for i, pos in enumerate(spiral)}
            HexChunk._spiral_cache[rings] = (spiral, pos_to_idx)
        
        self._spiral, self._pos_to_idx = HexChunk._spiral_cache[rings]
        
        # Core/halo boundary - indices < core_size are core
        self.core_size = core_size if core_size is not None else len(self._spiral)
        
        # Data arrays indexed by spiral order
        self.elevations = np.zeros(len(self._spiral))
        self.fields: dict[str, np.ndarray] = {}
        
        # Style per hex (optional override)
        self.hex_styles: list[StyleCSS | None] = [None] * len(self._spiral)
    
    def __len__(self):
        return len(self._spiral)
    
    # === Factory methods ===
    
    @classmethod
    def with_halo(cls, position: HexPosition, core_rings: int, halo_rings: int,
                  style: StyleCSS = None) -> 'HexChunk':
        """Create chunk with explicit core/halo split."""
        total_rings = core_rings + halo_rings
        core_size = cls.spiral_size(core_rings)
        return cls(position, total_rings, style, core_size=core_size)
    
    @staticmethod
    def spiral_size(rings: int) -> int:
        """Number of hexes in a spiral of N rings (including center)."""
        if rings == 0:
            return 1
        return 1 + 3 * rings * (rings + 1)
    
    # === Core/Halo properties ===
    
    @property
    def core_rings(self) -> int:
        """Number of rings in core (derived from core_size)."""
        # Inverse of spiral_size: solve 1 + 3*n*(n+1) = core_size
        # 3n² + 3n + 1 - core_size = 0
        if self.core_size <= 1:
            return 0
        n = int((np.sqrt(12 * self.core_size - 3) - 3) / 6)
        return n
    
    @property
    def halo_rings(self) -> int:
        """Number of rings in halo."""
        return self.rings - self.core_rings
    
    def is_core(self, idx: int) -> bool:
        """True if index is in core (not halo)."""
        return 0 <= idx < self.core_size
    
    def is_halo(self, idx: int) -> bool:
        """True if index is in halo."""
        return self.core_size <= idx < len(self)
    
    # === Iteration ===
    
    def iter_core(self) -> Iterator[int]:
        """Iterate over core indices only."""
        return iter(range(self.core_size))
    
    def iter_halo(self) -> Iterator[int]:
        """Iterate over halo indices only."""
        return iter(range(self.core_size, len(self)))
    
    def iter_all(self) -> Iterator[int]:
        """Iterate over all indices (core + halo)."""
        return iter(range(len(self)))
    
    def iter_core_with_pos(self) -> Iterator[tuple[int, HexPosition]]:
        """Yield (idx, local_pos) for core hexes."""
        for idx in self.iter_core():
            yield idx, self._spiral[idx]
    
    def iter_with_world(self) -> Iterator[tuple[int, HexPosition, HexPosition]]:
        """Yield (idx, local_pos, world_pos) for all hexes."""
        center = self.center_world
        for idx, local_pos in enumerate(self._spiral):
            yield idx, local_pos, center + local_pos
    
    # === World coordinate conversion ===
    
    @property
    def center_world(self) -> HexPosition:
        """World position of chunk center."""
        spacing = 2 * self.core_rings  # Match world_to_chunk spacing
        return HexPosition(
            self.position.q * spacing,
            self.position.r * spacing,
            self.position.s * spacing
        )
    
    @property
    def key(self) -> str:
        return f"{self.position.q}_{self.position.r}_{self.position.s}"
    
    # === Index conversion (local) ===
    
    def index_to_hexposition(self, idx: int) -> HexPosition | None:
        """Convert spiral index to local HexPosition (relative to chunk center)."""
        if 0 <= idx < len(self._spiral):
            return self._spiral[idx]
        return None
    
    def hexposition_to_index(self, pos: HexPosition) -> int:
        """Convert local HexPosition to spiral index. Returns -1 if not in chunk."""
        return self._pos_to_idx.get(pos, -1)
    
    # Shorthand aliases
    i2hp = index_to_hexposition
    hp2i = hexposition_to_index
    
    # === World coordinate conversion ===
    
    def index_to_world(self, idx: int) -> HexPosition | None:
        """Convert spiral index to world HexPosition."""
        local = self.index_to_hexposition(idx)
        if local is None:
            return None
        return self.center_world + local
    
    def world_to_index(self, world_pos: HexPosition) -> int:
        """Convert world HexPosition to spiral index. Returns -1 if not in chunk."""
        local = world_pos - self.center_world
        return self.hexposition_to_index(local)
    
    # === Neighbors ===
    
    def neighbors_of(self, idx: int, ring: int = 1) -> list[int]:
        """Get indices of neighbors within chunk. Filters out-of-bounds."""
        local_pos = self.index_to_hexposition(idx)
        if local_pos is None:
            return []
        
        neighbor_positions = local_pos.ring(ring)
        return [self.hexposition_to_index(p) for p in neighbor_positions 
                if self.hexposition_to_index(p) >= 0]
    
    def core_neighbors_of(self, idx: int) -> list[int]:
        """Get neighbors that are in core only."""
        return [n for n in self.neighbors_of(idx) if self.is_core(n)]
    
    # === Edge/Border helpers ===
    
    def edge_indices(self, direction: int) -> list[int]:
        """Indices along chunk edge facing direction (0-5)."""
        # Edge hexes are at max distance in that direction
        edge = []
        dir_hp = HexPosition.directions()[direction]
        for idx in self.iter_core():
            pos = self._spiral[idx]
            neighbor_pos = pos + dir_hp
            # If neighbor would be outside core, this is an edge hex
            if self.hexposition_to_index(neighbor_pos) < 0 or \
               not self.is_core(self.hexposition_to_index(neighbor_pos)):
                edge.append(idx)
        return edge
    
    # === Data operations ===
    
    def trim_to_core(self) -> 'HexChunk':
        """Return new chunk with only core data (no halo)."""
        new_chunk = HexChunk(self.position, self.core_rings, self.style)
        new_chunk.elevations = self.elevations[:new_chunk.core_size].copy()
        for name, data in self.fields.items():
            new_chunk.fields[name] = data[:new_chunk.core_size].copy()
        new_chunk.hex_styles = self.hex_styles[:new_chunk.core_size].copy()
        return new_chunk
    
    def copy_from(self, other: 'HexChunk', world_offset: HexPosition = None):
        """Copy overlapping data from another chunk.
        
        Args:
            other: Source chunk
            world_offset: If None, assumes chunks share coordinate system
        """
        for other_idx, other_local, other_world in other.iter_with_world():
            my_idx = self.world_to_index(other_world)
            if my_idx >= 0:
                self.elevations[my_idx] = other.elevations[other_idx]
                for name in other.fields:
                    if name not in self.fields:
                        self.fields[name] = np.zeros(len(self))
                    self.fields[name][my_idx] = other.fields[name][other_idx]
    
    # === Field management ===
    
    def add_field(self, name: str, default: float = 0.0):
        """Add a new data field."""
        self.fields[name] = np.full(len(self), default)
    
    def get_field(self, name: str) -> np.ndarray | None:
        """Get field array by name."""
        return self.fields.get(name)
    
    # === Rendering ===
    
    def to_hexes(self, radius: float, center: MapCord) -> list[Hex]:
        """Generate Hex objects for rendering."""
        hexes = []
        for idx, local_pos in enumerate(self._spiral):
            pixel = local_pos.pixel(radius, center)
            style = self.hex_styles[idx] or self.style
            hexes.append(Hex(radius, pixel, style))
        return hexes
    
    def render_svg(self, radius: float, center: MapCord, 
                   core_only: bool = False) -> str:
        """Render chunk to SVG string."""
        svg = ""
        indices = self.iter_core() if core_only else self.iter_all()
        for idx in indices:
            local_pos = self._spiral[idx]
            pixel = local_pos.pixel(radius, center)
            style = self.hex_styles[idx] or self.style
            hex_obj = Hex(radius, pixel, style)
            svg += "\t" + hex_obj.svg() + "\n"
        return svg
    
    def apply_styles(self, color_levels: list[StyleCSS], 
                     elevation_delta: float = 100,
                     sea_level: StyleCSS = None):
        """Apply styles based on elevation (like Terrain.colorMap)."""
        num_levels = len(color_levels)
        
        for i in range(len(self)):
            elev = self.elevations[i]
            if elev <= 0 and sea_level:
                self.hex_styles[i] = sea_level
            else:
                level = int(elev / elevation_delta)
                level = min(level, num_levels - 1)
                level = max(level, 0)
                self.hex_styles[i] = color_levels[level]


In [ ]:
#| export
@patch
def render_chunk(self: SVGBuilder, chunk: HexChunk, radius: float, 
                 center: MapCord, layer_name: str = "chunk"):
    """Render a HexChunk to a layer.
    chunk = HexChunk(HexPosition.origin(), rings=3)
chunk.elevations[0] = 500  # Center peak
chunk.elevations[1:7] = 300  # Ring 1
chunk.apply_styles(StyleCSS.elevations(), elevation_delta=100)

builder = SVGBuilder()
builder.width = 400
builder.height = 400
builder.render_chunk(chunk, radius=20, center=MapCord(200, 200))

    
    """
    # Add chunk's styles
    self.add_style(chunk.style)
    for style in chunk.hex_styles:
        if style:
            self.add_style(style)
    
    svg = chunk.render_svg(radius, center, self)
    self.adjust(layer_name, svg)


In [ ]:
def demo_hex_chunk():
    # Create a chunk at origin with 3 rings
    chunk = HexChunk(HexPosition.origin(), rings=3)
    
    # Set up some elevation data - peak in center, descending outward
    chunk.elevations[0] = 500  # Center
    
    # Ring 1 (indices 1-6)
    for i in range(1, 7):
        chunk.elevations[i] = 350
    
    # Ring 2 (indices 7-18)
    for i in range(7, 19):
        chunk.elevations[i] = 200
    
    # Ring 3 (indices 19-36)
    for i in range(19, len(chunk)):
        chunk.elevations[i] = 50
    
    # Apply terrain coloring
    colors = StyleCSS.elevations()
    chunk.apply_styles(colors, elevation_delta=100)
    
    # Build SVG
    builder = SVGBuilder()
    builder.width = 300
    builder.height = 300
    
    # Add elevation styles
    for style in colors:
        builder.add_style(style)
    
    builder.render_chunk(chunk, radius=25, center=MapCord(150, 150))
    
    print(f"Chunk size: {len(chunk)} hexes")
    print(f"Index 0 -> HexPosition: {chunk.i2hp(0)}")
    print(f"HexPosition(1,-1,0) -> index: {chunk.hp2i(HexPosition(1,-1,0))}")
    print(f"Neighbors of center: {chunk.neighbors_of(0)}")
    
    return builder.show()

demo_hex_chunk()
